In [1]:
import pandas as pd
import re
import numpy as np

# Pliki wejściowe
files = ["openalex_articles.csv", "openalex_manuscripts.csv"]
output_file = "openalex_articles_fixed.csv"

def clean_openalex_text(text):
    """
    1. Spłaszcza tekst / wiersze (usuwa entery/tabulatory).
    2. Usuwa konkretne tagi HTML/XML i ich atrybuty, zostawiając treść.
       Nie rusza nawiasów typu <1585-1675> ani <tekst>.
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Spłaszczanie
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. Usuwanie znanych tagów technicznych (struktura, formatowanie)
    # Wyjaśnienie regexa:
    # </? -> początek tagu otwieranego lub zamykanego
    # (html|body|head|...) -> lista nazw tagów do usunięcia
    # [^>]* -> dowolne atrybuty w środku tagu (np. class="...")
    # > -> koniec tagu
    # flag=re.I -> ignoruje wielkość liter (body i BODY)
    tags_to_remove = r'</?(html|head|body|!DOCTYPE|p|em|i|h\d?|r|scp|br|italic|sup|span)[^>]*>'
    text = re.sub(tags_to_remove, ' ', text, flags=re.I)
    
    return text.strip()

print("1. Wczytywanie i łączenie plików...")
try:
    # dtype=object zapobiega automatycznej, błędnej konwersji przy błędach w CSV
    df_list = [pd.read_csv(f, on_bad_lines='skip', low_memory=False) for f in files]
    df = pd.concat(df_list, ignore_index=True)
    print(f"Połączono. Łącznie wierszy na start: {len(df):,}")
except Exception as e:
    print(f"Błąd krytyczny: {e}")
    df = pd.DataFrame()

if not df.empty:
    # 2. Czyszczenie tekstów
    print("2. Czyszczenie tekstów (usuwanie tagów HTML, spłaszczanie)...")
    cols_to_clean = ['title', 'abstract', 'creator', 'subject', 'publisher']
    
    for col in cols_to_clean:
        if col in df.columns:
            # fillna('') potrzebne, by operować na stringach
            df[col] = df[col].fillna("").astype(str).apply(clean_openalex_text)
            # zamiana pustych stringów z powrotem na NaN dla łatwiejszego filtrowania
            df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)

    # 3. Naprawa roku (na Int64)
    print("3. Formatowanie roku (Int64)...")
    if 'year' in df.columns:
        # errors='coerce' zamieni błędne wpisy na NaN
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

    # 4. Logika filtrowania (usuwanie słabych rekordów)
    print("4. Filtrowanie (usuwanie słabych rekordów)...")
    initial_count = len(df)
    
    # Pomocnicze maski (True = brak danych)
    no_creator = df['creator'].isna()
    no_abstract = df['abstract'].isna()
    no_title = df['title'].isna()
    no_year = df['year'].isna()
    
    # Sprawdzenie długości tytułu (dla stringów) - bezpiecznie dla NaN
    ## jeśli tytuł jest NaN, length uznajemy za 0
    title_len = df['title'].astype(str).str.len()
    short_title = title_len <= 35
    
    # Warunek 1: Brak creator + abstract + year
    cond1 = no_creator & no_abstract & no_year
    
    # Warunek 2: Brak title + creator + abstract
    cond2 = no_title & no_creator & no_abstract
    
    # Warunek 3: Brak creator + abstract oraz tytuł istnieje, ale jest za krótki (<=35)
    ### zakładamy, że short_title łapie też puste tytuły, ale cond2 już je obsłużył, tu chodzi o te co maja tytuł, ale krótki
    cond3 = no_creator & no_abstract & short_title
    
    rows_to_drop = cond1 | cond2 | cond3
    
    df = df[~rows_to_drop]
    print(f"Usunięto {initial_count - len(df):,} rekordów spełniających warunki usunięcia.")

    # 5. Deduplikacja
    dedup_cols = ['title', 'creator']
    print(f"5. Deduplikacja wg: {dedup_cols}...")
    before_dedup = len(df)
    df = df.drop_duplicates(subset=dedup_cols, keep='first')
    print(f"Usunięto duplikatów: {before_dedup - len(df):,}")

    # 6. Zapis
    print(f"6. Zapisywanie do {output_file}...")
    df.to_csv(output_file, index=False, encoding='utf-8')
    
    # Podgląd
    print("\n--- Wynik ---")
    print(df[['title', 'year']].head(5))
    print(f"Liczba końcowa rekordów: {len(df):,}")


1. Wczytywanie i łączenie plików...
Połączono. Łącznie wierszy na start: 523,344
2. Czyszczenie tekstów (usuwanie tagów HTML, spłaszczanie)...


C:\Users\Greg Z\AppData\Local\Temp\ipykernel_28456\468643931.py:53: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)


3. Formatowanie roku (Int64)...
4. Filtrowanie (usuwanie słabych rekordów)...
Usunięto 2,794 rekordów spełniających warunki usunięcia.
5. Deduplikacja wg: ['title', 'creator']...
Usunięto duplikatów: 143,206
6. Zapisywanie do openalex_articles_fixed.csv...

--- Wynik ---
                                               title  year
0  EGYPT AND SYRIA IN THE FATIMID, AYYUBID AND MA...  1995
1               Ayyubid Cairo: A Topographical Study  1992
2  From Saladin to the Mongols: The Ayyubids of D...  1977
3  Women as Patrons of Religious Architecture in ...  1994
4          A History of the Ayyubid Sultans of Egypt  1983
Liczba końcowa rekordów: 377,344
